# Lab 7 — Train Your Own, or Borrow One

**Computer Vision · Session 7 — Bridge to Deep Learning**

> **Opening this in Colab:** download it from **https://classes.incortx.com/ComputerVision/session-07/lab.ipynb** , then in Colab choose
> **File → Upload notebook**. It also runs locally in Jupyter — it needs `torch`, `torchvision`,
> `scikit-learn` and `matplotlib`, all of which Colab already has.

Session 7 said a CNN is Session 2's convolution with the kernel found from data. This notebook does
that for real, twice, on the same small dataset:

| | Recipe | What you supply |
|---|---|---|
| **A** | **Train your own CNN** | 200 labelled images and a few seconds of training |
| **B** | **Borrow a trained one** | the same 200 images, and a model someone else trained on a million |

Both are scored on the **same 2,000 held-out images**, so the comparison is fair.

**Runtime:** about a minute of compute on a laptop CPU — 2 s to train the CNN, 22 s to extract the borrowed features — plus the downloads. Budget a few minutes on a free Colab CPU; **no GPU needed.**
**Downloads:** MNIST ≈ 11 MB · ResNet-18 weights ≈ 45 MB. (An optional Part E adds 170 MB.)

### How to use this notebook

Run the cells top to bottom. Each part ends with **TODOs** — small edits to the cell above it, with the
answer we measured written underneath, so you can check yourself.

**The numbers in this notebook are real.** Everything quoted here came from running this file. You may
see the last digit move (library versions, thread counts); if a number moves by more than a point or
two, something else changed and it is worth finding out what.

In [ ]:
import time
import numpy as np
import torch, torch.nn as nn, torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)

print('torch      ', torch.__version__)
print('torchvision', torchvision.__version__)
print('device     ', 'cuda' if torch.cuda.is_available() else 'cpu (this is fine — the lab is small)')

---
# Part 0 — The data, and why it is deliberately tiny

MNIST is 60,000 handwritten digits, 28×28, grey. **We are going to throw almost all of it away.**

Your project will have a few hundred photographs, not sixty thousand — so this lab keeps
**20 images per class (200 in total)** for training. The held-out set stays large (200 per class,
**2,000 images**) because a score measured on a handful of images is noise, not evidence.

In [ ]:
train_full = torchvision.datasets.MNIST('./data', train=True,  download=True)
test_full  = torchvision.datasets.MNIST('./data', train=False, download=True)

def take(images, labels, per_class, seed=0):
    """A balanced subset: `per_class` images of every digit, chosen at random."""
    rng, idx = np.random.default_rng(seed), []
    for c in range(10):
        idx += list(rng.choice(np.where(labels == c)[0], per_class, replace=False))
    idx = np.array(sorted(idx))
    return images[idx].astype(np.float32) / 255.0, labels[idx].astype(np.int64)

Xtr_all, ytr_all = train_full.data.numpy(), train_full.targets.numpy()
Xte_all, yte_all = test_full.data.numpy(),  test_full.targets.numpy()

X_train, y_train = take(Xtr_all, ytr_all, 20, seed=0)    # 200 images — all we get to learn from
X_test,  y_test  = take(Xte_all, yte_all, 200, seed=1)   # 2,000 images — never trained on

print('train   ', X_train.shape, ' held-out', X_test.shape)
print('classes ', np.bincount(y_train), '(20 of each, on purpose)')

In [ ]:
fig, axs = plt.subplots(2, 10, figsize=(13, 3))
for c in range(10):
    for row in range(2):
        ax = axs[row, c]
        ax.imshow(X_train[y_train == c][row], cmap='gray'); ax.axis('off')
    axs[0, c].set_title(str(c), fontsize=12)
fig.suptitle('Two of the twenty training images we have for each digit', y=1.02)
plt.show()

**Look at them before you model them.** Some of those 3s and 5s are ambiguous even to you — that is
the ceiling any model is working against, and it is a good habit to know what it looks like before
you start reading accuracy numbers.

---
# Part A — Train your own CNN

The architecture from the session: conv → pool, twice, then flatten and two dense layers.
Nothing here is special; the point is that **you can see every shape and every weight**.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 28x28 -> 14x14
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 14x14 ->  7x7
        )
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(7 * 7 * 32, 128), nn.ReLU(), nn.Linear(128, 10))

    def forward(self, x):
        return self.head(self.features(x))

# Walk the shapes the way slide 26 does — but let the code do the arithmetic.
model = SmallCNN()
x = torch.zeros(1, 1, 28, 28)
print(f'{"input":22s} {tuple(x.shape[1:])}')
for layer in list(model.features) + list(model.head):
    x = layer(x)
    print(f'{layer.__class__.__name__:22s} {tuple(x.shape[1:])}')

total = sum(p.numel() for p in model.parameters())
dense = sum(p.numel() for p in model.head.parameters())
print(f'\nparameters: {total:,} in total, {dense:,} of them ({100*dense/total:.0f}%) in the dense head')

**That percentage is the point of sub-topic C.** The convolutions see the whole image with a few
thousand weights, because each kernel is reused everywhere. The dense layer that follows the flatten
gives every one of the 1,568 numbers its own private weight, and swallows most of the model.

In [ ]:
def to_tensor(X, y):
    return torch.tensor(X).unsqueeze(1), torch.tensor(y)   # N,1,28,28

xt, yt = to_tensor(X_train, y_train)
xv, yv = to_tensor(X_test,  y_test)

def accuracy(model, x, y):
    model.eval()
    with torch.no_grad():
        return (model(x).argmax(1) == y).float().mean().item()

torch.manual_seed(SEED)
model = SmallCNN()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

curve, t0 = [], time.time()
for epoch in range(20):
    model.train()
    order = torch.randperm(len(xt))
    for i in range(0, len(order), 32):                      # mini-batches of 32
        batch = order[i:i + 32]
        opt.zero_grad()
        loss_fn(model(xt[batch]), yt[batch]).backward()     # the whole algorithm
        opt.step()
    curve.append((accuracy(model, xt, yt), accuracy(model, xv, yv)))
    if epoch % 5 == 4 or epoch == 0:
        print(f'epoch {epoch+1:2d} | train {curve[-1][0]:.3f} | held-out {curve[-1][1]:.3f}')

scratch_acc = curve[-1][1]
print(f'\ntrained in {time.time()-t0:.0f}s on {len(xt)} images')

In [ ]:
tr = [c[0] for c in curve]; te = [c[1] for c in curve]
plt.figure(figsize=(7.5, 3.6))
plt.plot(range(1, len(tr)+1), tr, 'o-', label='train (200 images)')
plt.plot(range(1, len(te)+1), te, 's-', label='held-out (2,000 images)')
plt.fill_between(range(1, len(tr)+1), te, tr, alpha=.12, color='crimson')
plt.xlabel('epoch'); plt.ylabel('accuracy'); plt.ylim(0, 1.02)
plt.title('The gap is the whole story'); plt.legend(); plt.grid(alpha=.3)
plt.show()
print(f'final: train {tr[-1]:.3f} · held-out {te[-1]:.3f} · gap {tr[-1]-te[-1]:.3f}')

**What we measured:** train **0.995**, held-out **0.854** — a 14-point gap after 20 epochs and about
two seconds of training.

The training score says the model memorised its 200 images perfectly. It did. That number is worth
nothing on its own, which is exactly the warning from sub-topic E: **a training score of 1.000 tells
you the model has enough capacity to memorise, and nothing about whether it learned anything.**

---
# Part B — Borrow a model somebody else trained

ResNet-18 was trained on ImageNet: about 1.2 million photographs, 1,000 classes, none of them a
handwritten digit. We are going to keep **everything it learned except its last layer**, and train
one linear layer of our own on top — the *freeze the backbone* row of the table on slide 45.

In [ ]:
weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
backbone = torchvision.models.resnet18(weights=weights)
backbone.fc = nn.Identity()          # cut off its 1,000-class classifier
backbone.eval()                      # frozen: we never call .backward() on it

print('parameters we are borrowing:', f'{sum(p.numel() for p in backbone.parameters()):,}')
print('parameters we will train   :', f'{512 * 10 + 10:,}  (one 512 -> 10 linear layer, fitted below)')

In [ ]:
# What did those borrowed weights actually learn? Look at the first layer.
k = backbone.conv1.weight.data.clone()          # 64 kernels, 3 channels, 7x7
k = (k - k.min()) / (k.max() - k.min())
fig, axs = plt.subplots(4, 16, figsize=(14, 3.6))
for ax, kernel in zip(axs.ravel(), k):
    ax.imshow(kernel.permute(1, 2, 0).numpy()); ax.axis('off')
fig.suptitle("ResNet-18's first layer — edges and colour blobs, learned from photographs", y=1.04)
plt.show()

**Compare that with slide 16.** Nobody wrote those kernels either. A network trained on cats, cars and
mushrooms arrived at edge detectors and colour blobs — the same things you built by hand in
Sessions 2–4. That is *why* borrowing works: the first layers of a vision model are not about the
task, they are about images.

In [ ]:
normalise = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

def features(X, size=128, batch=128):
    """Run the frozen backbone once and keep the 512 numbers it produces per image."""
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            b = torch.tensor(X[i:i+batch]).unsqueeze(1).repeat(1, 3, 1, 1)   # grey -> 3 channels
            b = torch.nn.functional.interpolate(b, size=size, mode='bilinear', align_corners=False)
            out.append(backbone(normalise(b)).numpy())
    return np.concatenate(out)

t0 = time.time()
F_train, F_test = features(X_train), features(X_test)
print(f'features: {F_train.shape} and {F_test.shape} in {time.time()-t0:.0f}s')

clf = LogisticRegression(max_iter=3000).fit(F_train, y_train)
transfer_acc = clf.score(F_test, y_test)
print(f'train {clf.score(F_train, y_train):.3f} | held-out {transfer_acc:.3f}')

**What we measured:** held-out **0.879** — against **0.854** from the CNN you trained yourself, on the
same 200 images.

Note what we did *not* do: no backpropagation through the backbone, no GPU, no tuning. The frozen
features were computed once and a linear model was fitted on top in under a second.

---
# Part C — Compare them honestly

In [ ]:
labels = ['your CNN\n(200 images)', 'frozen ResNet-18\n+ linear head']
scores = [scratch_acc, transfer_acc]
plt.figure(figsize=(6.4, 3.4))
bars = plt.bar(labels, scores, color=['#0E7C86', '#F97316'], width=.55)
for b, s in zip(bars, scores):
    plt.text(b.get_x() + b.get_width()/2, s + .01, f'{s:.3f}', ha='center', fontsize=13)
plt.ylim(0, 1.05); plt.ylabel('held-out accuracy (2,000 images)')
plt.title('Same 200 training images, same test set'); plt.show()
print(f'difference: {100*(transfer_acc - scratch_acc):+.1f} points')

### Read this result carefully — it is smaller than the internet promises

Transfer won by about **two and a half points** here (0.879 against 0.854). That is real, and free, but it is not the
tenfold miracle transfer learning is usually sold as. Two honest reasons:

1. **Digits are easy and unlike ImageNet.** A 28×28 grey digit has very little in common with a
   photograph of a dog, so a lot of what ResNet knows is irrelevant here. Its edge detectors still
   help; its "fur versus feathers" features do not.
2. **Training from scratch is genuinely fine on this task.** MNIST is the easiest benchmark in vision.
   Give the same small CNN 500 images instead of 200 and it reaches **0.895** on its own.

**The rule the session gives you still holds, and this measurement is how you check it:** run the
cheap baseline, run the transfer version, and let the held-out numbers decide. Part E repeats exactly
this comparison on a harder dataset, where the gap is much wider.

---
# Part D — Your turn

Each TODO is a small edit to a cell above. Measured answers are underneath, so you can check
yourself — try to predict before you run.

In [ ]:
# TODO 1: change `take(Xtr_all, ytr_all, 20, ...)` to 10 per class (100 images) and rerun
#         Part A and Part B. Which recipe loses more when the data is halved?
# TODO 2: go the other way — 50 per class (500 images). Does the gap between the two close?
# TODO 3: in `features()`, set size=64 instead of 128. It runs twice as fast. What does it cost?
# TODO 4: swap resnet18 for resnet34 (one word). Better? Slower? Worth it?
# TODO 5: print the confusion matrix of the transfer model
#         (from sklearn.metrics import confusion_matrix) — which digits does it mix up?
# TODO 6: one paragraph — a colleague has 300 photographs of two kinds of bolt and asks
#         which recipe to use. What do you tell them, and what would you measure first?

### Measured answers

| | Result |
|---|---|
| **1** · 100 images | your CNN **0.784** · transfer **0.840** — **transfer loses less**; that is the whole case for it |
| **2** · 500 images | your CNN **0.895** · transfer **0.918** — both improve, and the gap narrows to 2.3 points |
| **3** · 64 px | transfer drops to **0.852** at 200 images — the backbone was trained at 224, and shrinking the input throws away what it is looking for |
| **4** · resnet34 | **0.871 — slightly *worse* than resnet18's 0.879**, and 29 s of feature extraction instead of 21 s. **A bigger backbone is not automatically a better one** |
| **5** | one pair dominates: **5 predicted as 3, 25 times out of 200**. Then 8→3 and 2→8, 12 each. Anything with that open upper-left curve drifts towards 3 |
| **6** | Run both. It costs ten minutes, and the answer depends on their photographs, not on ours. Start with the frozen backbone, because it is the cheaper of the two to try. |

**The pattern across 1 and 2 is the one to remember: the fewer images you have, the more borrowing is
worth.** At 100 images it buys nearly 6 points; by 500 it is down to 2.

---
# Part E — Optional: the same two recipes on a harder problem

Everything above ran on digits because they download in seconds. Cats versus dogs is a fairer test of
transfer learning: real photographs, real backgrounds, and a task the ImageNet features actually know
something about.

**This cell downloads CIFAR-10 (≈ 170 MB)** and keeps only the cat and dog classes: 1,000 training
images, 1,000 held-out. Skip it if you are on a slow connection — the numbers are quoted below.

In [ ]:
tr = torchvision.datasets.CIFAR10('./data', train=True,  download=True)
te = torchvision.datasets.CIFAR10('./data', train=False, download=True)
CAT, DOG = 3, 5

def cats_and_dogs(ds, per_class):
    y = np.array(ds.targets); idx = []
    for c in (CAT, DOG):
        idx += list(np.where(y == c)[0][:per_class])
    idx = np.array(sorted(idx))
    return ds.data[idx].astype(np.float32) / 255.0, (y[idx] == DOG).astype(np.int64)

CX_train, Cy_train = cats_and_dogs(tr, 500)     # 1,000 images, 32x32 colour
CX_test,  Cy_test  = cats_and_dogs(te, 500)
print('train', CX_train.shape, '· held-out', CX_test.shape)

fig, axs = plt.subplots(1, 8, figsize=(13, 2))
for ax, i in zip(axs, [0, 1, 2, 3, 500, 501, 502, 503]):
    ax.imshow(CX_train[i]); ax.set_title('dog' if Cy_train[i] else 'cat', fontsize=11); ax.axis('off')
plt.show()

In [ ]:
class ColourCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(8 * 8 * 32, 128), nn.ReLU(), nn.Linear(128, 2))
    def forward(self, x):
        return self.head(self.features(x))

xt = torch.tensor(CX_train).permute(0, 3, 1, 2); yt = torch.tensor(Cy_train)
xv = torch.tensor(CX_test).permute(0, 3, 1, 2);  yv = torch.tensor(Cy_test)

torch.manual_seed(SEED)
cmodel = ColourCNN(); opt = torch.optim.Adam(cmodel.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(20):
    cmodel.train(); order = torch.randperm(len(xt))
    for i in range(0, len(order), 32):
        b = order[i:i+32]; opt.zero_grad()
        loss_fn(cmodel(xt[b]), yt[b]).backward(); opt.step()
    if epoch % 5 == 4:
        print(f'epoch {epoch+1:2d} | train {accuracy(cmodel, xt, yt):.3f} | held-out {accuracy(cmodel, xv, yv):.3f}')

def colour_features(X, size=128, batch=64):
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            b = torch.tensor(X[i:i+batch]).permute(0, 3, 1, 2)
            b = torch.nn.functional.interpolate(b, size=size, mode='bilinear', align_corners=False)
            out.append(backbone(normalise(b)).numpy())
    return np.concatenate(out)

CF_train, CF_test = colour_features(CX_train), colour_features(CX_test)
cclf = LogisticRegression(max_iter=3000).fit(CF_train, Cy_train)
print(f'\nyour CNN        held-out {accuracy(cmodel, xv, yv):.3f}')
print(f'frozen ResNet-18 held-out {cclf.score(CF_test, Cy_test):.3f}')

### What to expect here

> ⚠️ **We have not run this part for you.** Every other number in this notebook came off our own
> screen; this one has not, so it is not quoted. Run the two cells above and write down what you
> get — that is the whole habit this course is trying to build.

Cats and dogs are photographs of animals, which is **exactly** what ImageNet is full of — so the
borrowed features are far more relevant here than they were on digits, and the gap should be much
wider than the two and a half points you measured in Part C.

That is why the first question to ask about a small vision project is not "which architecture" but
**"whose features can I start from?"** — and why the answer is worth measuring rather than assuming.

---
### What to take away

1. **Both recipes are three cells long.** Neither is hard; the work is in measuring, not in coding.
2. **A training score of 1.000 means nothing.** Only the held-out number counts, and only if the split
   was made before anything was fitted.
3. **Borrowing pays most when you have least.** 100 images: +5.6 points. 500 images: +2.3.
4. **The size of the win depends on how close your images are to what the backbone saw.** Digits: a
   couple of points. Photographs of animals: far more.